# Chapter 23 · PyTorch does the bookkeeping
**Deck 4 · Live demo.** Rebuild Chapter 21’s chain and Chapter 22’s network in PyTorch, and match every number.

**You will:** use tensors with requires_grad · call backward() · train with the same five moves · prove the numbers match.

In [ ]:
import torch
import numpy as np
torch.set_default_dtype(torch.float64)
print(torch.__version__)

## 1 · Your prediction
After `loss.backward()`, what will `w1.grad` print?

In [ ]:
my_prediction = -24

## 2 · The chain, with autograd

In [ ]:
x, y = torch.tensor(2.0), torch.tensor(5.0)
w1 = torch.tensor(0.5, requires_grad=True)
w2 = torch.tensor(3.0, requires_grad=True)

loss = (w2 * torch.relu(w1 * x) - y) ** 2   # the forward pass is recorded
loss.backward()                              # the blame game, automatically
print('loss', loss.item(), '· w1.grad', w1.grad.item(), '· w2.grad', w2.grad.item())

## 3 · The 2-2-1 network, same fixed start

In [ ]:
X = torch.tensor([[0., 0.], [1., 1.], [1., 0.], [0., 1.]])
Y = torch.tensor([1., 1., 0., 0.])

model = torch.nn.Sequential(torch.nn.Linear(2, 2), torch.nn.Sigmoid(), torch.nn.Linear(2, 1), torch.nn.Sigmoid())
with torch.no_grad():                         # PyTorch stores weights as [out, in]
    model[0].weight.copy_(torch.tensor([[0.5, 0.4], [-0.5, -0.6]]))
    model[0].bias.copy_(torch.tensor([0.1, 0.2]))
    model[2].weight.copy_(torch.tensor([[0.6, 0.7]]))
    model[2].bias.copy_(torch.tensor([0.0]))

loss_fn = torch.nn.BCELoss()
opt = torch.optim.SGD(model.parameters(), lr=1.0)
print('start loss', round(loss_fn(model(X).squeeze(), Y).item(), 3))

## 4 · The training loop: our five moves

In [ ]:
table = {}
for epoch in range(3001):
    pred = model(X).squeeze()        # forward
    loss = loss_fn(pred, Y)          # loss
    if epoch in (0, 100, 300, 500, 1000, 3000):
        table[epoch] = round(loss.item(), 3)
    opt.zero_grad()                  # clear last step's slopes
    loss.backward()                  # every slope
    opt.step()                       # every weight moves
print(table)

## 5 · PROOF

In [ ]:
assert loss_fn is not None
x2, y2 = torch.tensor(2.0), torch.tensor(5.0)
a = torch.tensor(0.5, requires_grad=True); b = torch.tensor(3.0, requires_grad=True)
l = (b * torch.relu(a * x2) - y2) ** 2; l.backward()
assert (l.item(), a.grad.item(), b.grad.item()) == (4.0, -24.0, -4.0), 'same as Chapter 21 by hand'
assert list(table.values()) == [0.748, 0.681, 0.472, 0.085, 0.017, 0.004], 'same loss table as NumPy (Chapter 22)'
assert my_prediction == a.grad.item()
print('✓ proof: autograd = the blame game; PyTorch matches NumPy to 3 decimals')

## 6 · Open the box

In [ ]:
with torch.no_grad():
    print('hidden activations:', torch.sigmoid(model[0](X)).numpy().round(2))
    print('P(pass):', model(X).squeeze().numpy().round(3))

## 7 · Your turn
Swap the hidden Sigmoid for ReLU (`torch.nn.ReLU()`). Train again. What changes?

In [ ]:
# TODO

## Recap
`loss.backward()` is the blame game. `opt.step()` is w ← w − η × slope. PyTorch only keeps the books.